In [3]:
import sys
import os

# add project root (fairvit-med) to path
PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

print("Project root added:", PROJECT_ROOT)


Project root added: e:\fairvit-med


In [4]:
import sys
import os

# add project root
sys.path.insert(0, os.path.abspath(".."))

import torch

from src.models.vit_model import FairViTMed
from src.dataset import ChestXrayDataset
from src.dataloaders import get_dataloaders


In [5]:
import src
print(src.__file__)


e:\fairvit-med\src\__init__.py


In [6]:
import os
import torch

from src.models.vit_model import FairViTMed
from src.dataloaders import get_dataloaders
from src.training.train import train_one_epoch, evaluate, criterion

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



Device: cuda


In [11]:
model = FairViTMed(num_classes=1)
model = model.to(device)

print("Model loaded")



Model loaded


In [10]:
print(model)

FairViTMed(
  (vit): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): EncoderBlock(
          (ln_1): LayerN

In [12]:
import torch
import torch.nn as nn

# Freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the classification head
for param in model.vit.heads.head.parameters():
    param.requires_grad = True

print("Classification head unfrozen ✅")

# Verify trainable parameters
print("\nTrainable parameters:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

# Create optimizer only for trainable params
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

print("\nOptimizer ready ✅")

Classification head unfrozen ✅

Trainable parameters:
vit.heads.head.weight
vit.heads.head.bias

Optimizer ready ✅


In [12]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

print("Optimizer ready")


Optimizer ready


In [14]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nProject tree (top level):")
print(os.listdir("."))



Current working directory:
e:\fairvit-med\notebooks

Project tree (top level):
['baseline_results.md', 'data_pipeline.md', 'eda.ipynb', 'eda_findings.md', 'model_architecture.md', 'model_test.ipynb', 'preprocessing.md', 'split.ipynb', 'train_baseline.ipynb', 'transforms.ipynb']


In [17]:
import os
print(os.getcwd())


e:\fairvit-med\notebooks


In [18]:
os.chdir(r"E:\fairvit-med")
print(os.getcwd())


E:\fairvit-med


In [19]:
print(os.listdir("."))


['.venv', '.vscode', 'check_fairness.py', 'create_labels.py', 'data', 'fairvit-env', 'inspect_data.py', 'models', 'notebooks', 'README.md', 'requirements.txt', 'src', 'test.py', 'test_gpu.py']


In [20]:
print(os.listdir("data"))


['processed', 'raw']


In [21]:
for root, dirs, files in os.walk("data"):
    for f in files:
        if f.endswith(".csv"):
            print(os.path.join(root, f))


data\processed\metadata.csv
data\processed\train_metadata.csv
data\processed\val_metadata.csv
data\raw\Data_Entry_2017.csv


In [22]:
train_loader, val_loader = get_dataloaders(
    train_csv="data/processed/train_metadata.csv",
    val_csv="data/processed/val_metadata.csv",
    image_dir="data/raw/images",
    batch_size=4
)

print("Dataloaders ready ✅")


Dataloaders ready ✅


In [23]:
images, labels, sexes = next(iter(train_loader))
print(images.shape)
print(labels.shape)
print(type(sexes), sexes[:5])


torch.Size([4, 3, 224, 224])
torch.Size([4])
<class 'tuple'> ('M', 'F', 'M', 'F')


In [ ]:
EPOCHS = 3

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_auc = evaluate(model, val_loader, device)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f}")


Epoch 1/3 | Loss: 0.0699 | Val AUC: 0.5491
Epoch 2/3 | Loss: 0.0693 | Val AUC: 0.5537
Epoch 3/3 | Loss: 0.0692 | Val AUC: 0.4416


In [25]:
import os
os.makedirs("models", exist_ok=True)

torch.save(model.state_dict(), "models/baseline_vit.pt")
print("Baseline model saved ✅")


Baseline model saved ✅


In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_positive = 287
num_negative = 22424 - 287

pos_weight_value = num_negative / num_positive

pos_weight = torch.tensor([pos_weight_value]).to(device)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print("pos_weight set successfully ✅")


pos_weight set successfully ✅


In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
import sys
import os

# Add project root to Python path
sys.path.append(os.path.abspath("."))

print("Project root added ")

Project root added 


In [5]:
import os
print(os.getcwd())
print(os.listdir())

e:\fairvit-med\notebooks
['data_pipeline.md', 'eda.ipynb', 'eda_findings.md', 'fairness_eval.ipynb', 'model_test.ipynb', 'preprocessing.md', 'split.ipynb', 'train_baseline.ipynb', 'transforms.ipynb']


In [7]:
import sys
import os

sys.path.append(os.path.abspath("."))  # since train notebook is in root
print("Project root added ")

Project root added 


In [13]:
import sys
import os

# Go to project root (one level up from notebooks)
project_root = os.path.abspath("..")
sys.path.append(project_root)

print("Project root added:", project_root)

Project root added: e:\fairvit-med


In [14]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score

from src.models.vit_model import FairViTMed
from src.dataloaders import get_dataloaders

In [16]:
train_loader, val_loader = get_dataloaders(
    train_csv="../data/processed/train_metadata.csv",
    val_csv="../data/processed/val_metadata.csv",
    image_dir="../data/raw/images",
    batch_size=4
)

print("Dataloaders ready ✅")

Dataloaders ready ✅


In [18]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [19]:
num_positive = 287
num_negative = 22424 - 287

pos_weight_value = num_negative / num_positive
pos_weight = torch.tensor([pos_weight_value]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print("Using pos_weight:", pos_weight_value)

Using pos_weight: 77.13240418118467


In [21]:
from src.models.vit_model import FairViTMed

model = FairViTMed(num_classes=1)
model = model.to(device)

print("Model loaded ✅")

Model loaded ✅


In [23]:
import torch.optim as optim
import torch.nn as nn

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4)
print("Optimizer ready ✅")

# Loss function (with pos_weight if using it)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print("Loss ready ✅")

Optimizer ready ✅
Loss ready ✅


In [1]:
import sys
import torch
import torch.nn as nn
import torch.optim as optim

sys.path.append("..")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

from src.models.vit_model import FairViTMed
from src.dataloaders import get_dataloaders

# Model
model = FairViTMed(num_classes=1).to(device)

# Dataloaders
train_loader, val_loader = get_dataloaders(
    train_csv="../data/processed/train_metadata.csv",
    val_csv="../data/processed/val_metadata.csv",
    image_dir="../data/raw/images",
    batch_size=4
)

# Class imbalance
num_positive = 287
num_negative = 22424 - 287
pos_weight_value = num_negative / num_positive
pos_weight = torch.tensor([pos_weight_value]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

print("Everything ready ✅")

Device: cuda
Everything ready ✅


In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels, _ in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    # ---- Validation ----
    model.eval()
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels, _ in val_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            probs = torch.sigmoid(outputs)

            val_preds.extend(probs.cpu().numpy().flatten())
            val_labels.extend(labels.cpu().numpy().flatten())

    val_auc = roc_auc_score(val_labels, val_preds)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss: {avg_loss:.4f} "
          f"Val AUC: {val_auc:.4f}")

Epoch [1/10] Loss: 1.4209 Val AUC: 0.5534
Epoch [2/10] Loss: 1.3829 Val AUC: 0.5495


In [4]:
for epoch in range(1):  # only 1 epoch for testing
    model.train()

    for images, labels, sex in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        print("Images device:", images.device)
        print("Model device:", next(model.parameters()).device)

        break  # stop after first batch

    break  # stop after first epoch

Images device: cuda:0
Model device: cuda:0


In [6]:
num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for batch_idx, (images, labels, _) in enumerate(train_loader):

        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # 🔹 Print progress every 100 batches
        if batch_idx % 100 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss {loss.item():.4f}")

    avg_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels, _ in val_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            probs = torch.sigmoid(outputs)

            val_preds.extend(probs.cpu().numpy().flatten())
            val_labels.extend(labels.cpu().numpy().flatten())

    val_auc = roc_auc_score(val_labels, val_preds)

    print(f"\nEpoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f} Val AUC: {val_auc:.4f}\n")

Epoch 1 | Batch 0/22424 | Loss 0.5122
Epoch 1 | Batch 100/22424 | Loss 0.1854
Epoch 1 | Batch 200/22424 | Loss 0.4347
Epoch 1 | Batch 300/22424 | Loss 0.3736
Epoch 1 | Batch 400/22424 | Loss 0.4160
Epoch 1 | Batch 500/22424 | Loss 0.5380
Epoch 1 | Batch 600/22424 | Loss 0.6079


KeyboardInterrupt: 